In [ ]:
##%%
import os
import sys
import pandas as pd

import sys
import os
project_root = os.path.abspath(os.path.join(os.path.join(os.path.join(os.getcwd(), '..'), '..'), '..')) 
if project_root not in sys.path:
    sys.path.append(project_root)

from notebooks.consts import OLIGO_CSV_PROCESSED
from notebooks.features.feature_extraction import save_feature

# --- Setup Paths ---
current_dir = os.getcwd()
sys.path.append(os.path.join(current_dir, 'OligoAI'))
from run_inference import run_inference

# --- Configuration ---
BATCH_SIZE = 500
NUM_WORKERS = 8
MODEL_PATH = "OligoAI/OligoAI_11_09_25.ckpt"
TEMP_INPUT_PATH = "OligoAI/data/temp_tauso_oligo_data.csv"

# 1. Load the database
oligo_data = pd.read_csv(OLIGO_CSV_PROCESSED)

# --- Rename Schema ---
rename_back_scheme = {
    "Sequence": "aso_sequence_5_to_3",
    "Cell_line": "cell_line",
    "Cell line organism": "cell_line_species",
    "Inhibition(%)": "inhibition_percent",
    "ASO_volume(nM)": "dosage",
}

# Duplicate columns for compatibility
for old_col, new_col in rename_back_scheme.items():
    if old_col in oligo_data.columns:
        # Create the new column as a direct copy of the old one
        oligo_data[new_col] = oligo_data[old_col]
       

# 2. Save temporarily for OligoAI inference
oligo_data.to_csv(TEMP_INPUT_PATH, index=False)

# 3. Run Inference
print("Running OligoAI inference to extract features...")
results_df = run_inference(
    model_checkpoint_path=MODEL_PATH,
    data_path=TEMP_INPUT_PATH,
    device="cuda",
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS
)

# 4. Extract the prediction column dynamically to avoid hardcoded column names
pred_cols = [c for c in results_df.columns if c.startswith('predicted_')]

if pred_cols:
    prediction_col_name = pred_cols[0]
    oligo_data['oligo_ai_score'] = results_df[prediction_col_name]
    
    # 5. Save the new feature to your database
    save_feature(oligo_data, 'oligo_ai_score', version='oligo')
    print("OligoAI feature successfully saved.")
else:
    print("Error: Could not locate the prediction column in the output dataframe.")

# --- Cleanup ---
files_to_remove = [
    TEMP_INPUT_PATH,
    TEMP_INPUT_PATH.replace('.csv', '.with_predictions.csv')
]

for f in files_to_remove:
    if os.path.exists(f):
        try:
            os.remove(f)
        except Exception:
            pass